# 03 · 领域事件、Replay 与轻量 CQRS

目标：把不可变事件折叠成状态，体验 expected version 乐观并发，并从同一事件构建读模型。对应文档：`03_architecture_patterns/3.4-event-driven-and-cqrs.md`。

## 从 Redux 迁移

`ReplayAccount(events)` 很像 reducer：历史事件按顺序折叠成状态。区别是后端事件可能成为长期事实，需要版本并发控制、schema 演进和幂等消费。

In [ ]:
import (
    "errors"
    "fmt"
    "sync"
)

var ErrVersionConflict = errors.New("event stream version conflict")

type AccountEvent struct {
    Kind   string
    Amount int
}

type Account struct {
    Balance int
    Version int
}

func ReplayAccount(events []AccountEvent) Account {
    var account Account
    for _, event := range events {
        switch event.Kind {
        case "deposited":
            account.Balance += event.Amount
        case "withdrawn":
            account.Balance -= event.Amount
        }
        account.Version++
    }
    return account
}

type MemoryEventStore struct {
    mu      sync.Mutex
    streams map[string][]AccountEvent
}

func NewMemoryEventStore() *MemoryEventStore {
    return &MemoryEventStore{streams: make(map[string][]AccountEvent)}
}

func (s *MemoryEventStore) Append(stream string, expectedVersion int, events ...AccountEvent) error {
    s.mu.Lock()
    defer s.mu.Unlock()
    if len(s.streams[stream]) != expectedVersion {
        return fmt.Errorf("append %s: %w", stream, ErrVersionConflict)
    }
    s.streams[stream] = append(s.streams[stream], events...)
    return nil
}

func (s *MemoryEventStore) Load(stream string) []AccountEvent {
    s.mu.Lock()
    defer s.mu.Unlock()
    return append([]AccountEvent(nil), s.streams[stream]...)
}

func SeedAccount() (*MemoryEventStore, Account) {
    store := NewMemoryEventStore()
    _ = store.Append("account-1", 0,
        AccountEvent{Kind: "deposited", Amount: 100},
        AccountEvent{Kind: "withdrawn", Amount: 30},
    )
    return store, ReplayAccount(store.Load("account-1"))
}

## 实验一：Replay

存入 100 的 deposit 和 30 的 withdrawal。先预测余额与 stream version，再运行。Replay 必须是确定性的，不能读取当前时间或随机数。

In [ ]:
%%
eventStore, account := SeedAccount()
history := eventStore.Load("account-1")
fmt.Printf("events=%d balance=%d version=%d\n", len(history), account.Balance, account.Version)

## 实验二：乐观并发

客户端以 expected version 0 写入已经有两个事件的 stream，应得到稳定冲突错误。重新加载 version 后再写入才安全。

In [ ]:
%%
eventStore, account := SeedAccount()
conflictErr := eventStore.Append("account-1", 0, AccountEvent{Kind: "deposited", Amount: 50})
fmt.Println("并发版本冲突:", errors.Is(conflictErr, ErrVersionConflict))

appendErr := eventStore.Append("account-1", account.Version, AccountEvent{Kind: "deposited", Amount: 50})
updated := ReplayAccount(eventStore.Load("account-1"))
fmt.Printf("appendErr=%v updatedBalance=%d updatedVersion=%d\n", appendErr, updated.Balance, updated.Version)

In [ ]:
type BalanceView struct {
    Deposits    int
    Withdrawals int
    Balance     int
}

func ProjectBalance(events []AccountEvent) BalanceView {
    var view BalanceView
    for _, event := range events {
        if event.Kind == "deposited" {
            view.Deposits++
            view.Balance += event.Amount
        }
        if event.Kind == "withdrawn" {
            view.Withdrawals++
            view.Balance -= event.Amount
        }
    }
    return view
}

In [ ]:
%%
projectionStore, _ := SeedAccount()
view := ProjectBalance(projectionStore.Load("account-1"))
fmt.Printf("CQRS read model: %+v\n", view)

写模型保护业务不变量，读模型针对查询预计算。两者异步分离时会出现最终一致窗口，调用方必须知道刚写入的数据何时可见。

In [ ]:
!*cd "$GO_LEARNING_LABS" && go test ./agent -run TestSessionRecordsToolCallFlowInOrder -v

## 动手挑战

1. 增加 `transfer_failed` 事件并保持旧事件可 replay。
2. 模拟两个 writer 使用同一 expected version。
3. 为事件增加 ID，并让投影端幂等去重。

## 复盘

- 领域事件和事件溯源有什么区别？
- Outbox 为什么仍可能重复投递？
- 什么时候轻量 CQRS 只需分开方法，而不需要两套数据库？